**NB** : **Pour le retard (stunting)**

# Librairies

### Lecture et visualisation des données

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Separation et validation

In [2]:
from sklearn.model_selection import (
    train_test_split, # separer train/test une seule fois
    StratifiedKFold, # validation croisee stratifiee
    cross_validate, # validation croisee avec plusieurs metriques + temps
    GridSearchCV, # recherche exhaustive des meilleurs hyperparametres
)

### Preparation des donnees

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder# Pour encodage
from sklearn.compose import ColumnTransformer
from sklearn.base import clone # Verification apres encodage

### Evaluation

In [4]:
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    fbeta_score,
    make_scorer,
    recall_score,
    roc_auc_score,
    log_loss,
    average_precision_score
)

from sklearn.inspection import permutation_importance

### Modele choisie

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [6]:
# Fixer le random_state
RANDOM_STATE = 42

In [7]:
df = pd.read_parquet('../data/clean/model_stunting.parquet')

print(f"Taille ; {df.shape}")

Taille ; (4864, 15)


In [8]:
df.head()

,age_mois,region,edu_mere,electricite,cuisiniere,toilettes,sexe_f,acte_naissance,eau_amelioree,animaux,stunt,grappe,menage,id_menage,ligne_enfant
0,33,Maritime,Primaire,Réseau,Feu ouvert,Chasse d'eau,1,1.0,0.0,0,0,1,7,1-7,5
1,41,Maritime,Primaire,Réseau,Combustible solide (cuisinière),Latrine améliorée,0,1.0,1.0,0,0,1,15,1-15,2
2,12,Maritime,Aucune,Réseau,Combustible solide (cuisinière),Non améliorée,1,0.0,1.0,0,0,1,18,1-18,6
3,22,Maritime,Aucune,Réseau,Combustible solide (cuisinière),Non améliorée,0,1.0,1.0,0,0,2,1,2-1,4
4,3,Maritime,Secondaire ou plus,Réseau,Combustible solide (cuisinière),Latrine améliorée,0,1.0,1.0,0,0,2,2,2-2,2


# Features / label

In [9]:
df.columns

Index(['age_mois', 'region', 'edu_mere', 'electricite', 'cuisiniere',
       'toilettes', 'sexe_f', 'acte_naissance', 'eau_amelioree', 'animaux',
       'stunt', 'grappe', 'menage', 'id_menage', 'ligne_enfant'],
      dtype='object')

In [10]:
NON_ENTREES = ["grappe", "menage", "id_menage", "ligne_enfant", 'stunt']

In [11]:
X = df.drop(columns=NON_ENTREES)# Entrees (features)
y = df['stunt'] # Sortie (label)

In [12]:
print(f"\nVérification : {X.columns.tolist()}")


Vérification : ['age_mois', 'region', 'edu_mere', 'electricite', 'cuisiniere', 'toilettes', 'sexe_f', 'acte_naissance', 'eau_amelioree', 'animaux']


In [13]:
y.unique()

array([0, 1], dtype=int8)

In [14]:
X.head()

,age_mois,region,edu_mere,electricite,cuisiniere,toilettes,sexe_f,acte_naissance,eau_amelioree,animaux
0,33,Maritime,Primaire,Réseau,Feu ouvert,Chasse d'eau,1,1.0,0.0,0
1,41,Maritime,Primaire,Réseau,Combustible solide (cuisinière),Latrine améliorée,0,1.0,1.0,0
2,12,Maritime,Aucune,Réseau,Combustible solide (cuisinière),Non améliorée,1,0.0,1.0,0
3,22,Maritime,Aucune,Réseau,Combustible solide (cuisinière),Non améliorée,0,1.0,1.0,0
4,3,Maritime,Secondaire ou plus,Réseau,Combustible solide (cuisinière),Latrine améliorée,0,1.0,1.0,0


# Train_test_split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=RANDOM_STATE,
    test_size=0.2,
    stratify=y
)

# Repartition 

print(f"Train : {X_train.shape[0]} observations")
print(f"Test  : {X_test.shape[0]} observations")

print("\nProportions des classes (train) :")
print(pd.Series(y_train).value_counts(normalize=True).sort_index())

print("\nProportions des classes (test) :")
print(pd.Series(y_test).value_counts(normalize=True).sort_index())

Train : 3891 observations
Test  : 973 observations

Proportions des classes (train) :
stunt
0    0.756104
1    0.243896
Name: proportion, dtype: float64

Proportions des classes (test) :
stunt
0    0.756423
1    0.243577
Name: proportion, dtype: float64


# Preprocessor

In [16]:
X['eau_amelioree'].unique()

array([0., 1.])

In [17]:
print("\n")
print(f"{30*'-'}Colonnes numériques{30*'-'}")
print("\n")
for c in X.select_dtypes(include='number').columns :
    print(f"{c}{10*' - '}{df[c].nunique()}")
    print()

print("\n")

print(f"{30*'-'}Colonnes non-numériques{30*'-'}")
print("\n")
for c in X.select_dtypes(include='object').columns :
    print(f"{c}{10*' - '}{df[c].unique()}{10*' - '}{df[c].nunique()}")
    print()



------------------------------Colonnes numériques------------------------------


age_mois -  -  -  -  -  -  -  -  -  - 60

sexe_f -  -  -  -  -  -  -  -  -  - 2

acte_naissance -  -  -  -  -  -  -  -  -  - 2

eau_amelioree -  -  -  -  -  -  -  -  -  - 2

animaux -  -  -  -  -  -  -  -  -  - 2



------------------------------Colonnes non-numériques------------------------------


region -  -  -  -  -  -  -  -  -  - ['Maritime' 'Plateaux' 'Centrale' 'Kara' 'Savanes' 'Lomé Commune'
 'Golfe Urbain'] -  -  -  -  -  -  -  -  -  - 7

edu_mere -  -  -  -  -  -  -  -  -  - ['Primaire' 'Aucune' 'Secondaire ou plus'] -  -  -  -  -  -  -  -  -  - 3

electricite -  -  -  -  -  -  -  -  -  - ['Réseau' 'Non' 'Hors réseau'] -  -  -  -  -  -  -  -  -  - 3

cuisiniere -  -  -  -  -  -  -  -  -  - ['Feu ouvert' 'Combustible solide (cuisinière)'
 'Électricité, gaz ou liquide' 'Pas de repas préparé'] -  -  -  -  -  -  -  -  -  - 4

toilettes -  -  -  -  -  -  -  -  -  - ["Chasse d'eau" 'Latrine amélior

**Rappel des ordres (du plus faible au plus fort), pour plus tard :**
- `edu_mere` : Aucune, Primaire, Secondaire ou plus
- `electricite` : Non, Hors réseau, Réseau
- `toilettes` : Aucune, Non améliorée, Latrine améliorée, Chasse d'eau
- `cuisiniere` : Pas de repas préparé, Feu ouvert, Combustible solide (cuisinière), Électricité, gaz ou liquide


**Rappel des binaires, pour plus tard :**
- `0` : non
- `1` : oui

In [18]:
numeric_features = ['age_mois']

nominal_features = ['region']

ordinal_features = ['edu_mere', 'cuisniere', 'electricite', 'toilettes']

binaires = ['sexe_f', 'acte_naissance', 'eau_amelioree', 'animaux']

edu_order = ['Aucune', 'Primaire', 'Secondaire ou plus']
cuisine_order = ['Pas de repas préparé', 'Feu ouvert', 'Combustible solide (cuisinière)', 'Électricité, gaz ou liquide']
elec_order = ['Non', 'Hors réseau', 'Réseau']
toilette_order = ['Aucune', 'Non améliorée', 'Latrine améliorée', 'Chasse d\'eau']

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        # Numeriques : standardisation
        ("num", StandardScaler(), numeric_features),

        # Nominales : sparse_output=False obligatoire avec la sortie pandas
        ("nom", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False), nominal_features),

        # Ordinale : encodage 0, 1, 2
        ("ord", OrdinalEncoder(categories=[edu_order, cuisine_order, elec_order, toilette_order]), ordinal_features),

        # Binaire : passe tel quel
        ("bin", "passthrough", binaires),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
preprocessor.set_output(transform="pandas")

ColumnTransformer(transformers=[('num', StandardScaler(), ['age_mois']),
                                ('nom',
                                 OneHotEncoder(drop='if_binary',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['region']),
                                ('ord',
                                 OrdinalEncoder(categories=[['Aucune',
                                                             'Primaire',
                                                             'Secondaire ou '
                                                             'plus'],
                                                            ['Pas de repas '
                                                             'préparé',
                                                             'Feu ouvert',
                                                             'Combustible '
                                                             'solide '
                                                             '(cuisinière)',
                                                             'Électricité, gaz '
                                                             'ou liquide'],
                                                            ['Non',
                                                             'Hors réseau',
                                                             'Réseau'],
                                                            ['Aucune',
                                                             'Non améliorée',
                                                             'Latrine '
                                                             'améliorée',
                                                             "Chasse d'eau"]]),
                                 ['edu_mere', 'cuisniere', 'electricite',
                                  'toilettes']),
                                ('bin', 'passthrough',
                                 ['sexe_f', 'acte_naissance', 'eau_amelioree',
                                  'animaux'])],
                  verbose_feature_names_out=False)

# Validation croisée

In [20]:


# Dictionnaire des modèles
models = {
    "Regression logistique": LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'
    ),
    "Arbre de decision": DecisionTreeClassifier(
        random_state=RANDOM_STATE, class_weight='balanced'
    ),
}

# Validation croisée stratifiée
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Dictionnaire des métriques
scoring = {
    'recall': 'recall',
    'auc_roc': 'roc_auc',             
    'log_loss': 'neg_log_loss',       
    'pr_auc': 'average_precision',     
}

results = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model),
    ])

    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    results[name] = scores

    print(f"{name:22s} -> "
          f"rec = {scores['test_recall'].mean():.3f} | "
          f"auc_roc = {scores['test_auc_roc'].mean():.3f} | "
          f"log_loss = {-scores['test_log_loss'].mean():.3f} | " 
          f"pr_auc = {scores['test_pr_auc'].mean():.3f} | "
          f"fit = {scores['fit_time'].mean():.2f}s")


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\HP ELITEBOOK\anaconda3\Lib\site-packages\pandas\core\indexes\base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'cuisniere'

The above exception was the direct cause of the foll